# Prepare State And Amplitudes in native FlagQuantum

Complete local computational Notebook workflow: original nonuniform probability vector, seeded sampling and probability reconstruction; host-generated signed linear amplitudes, normalization, native synthesis, raw complex state reconstruction and phase-aligned presentation. Local execution replaces cloud calls and plain Python arrays replace dataframes; no hardware result is claimed.

Ordinary host Python generates and normalizes the evenly spaced amplitude vector, then supplies immutable classical_inputs. Root preparation consumes bound vectors through existing register lowering. Independent acceptance compares raw absolute amplitudes and separately checks phase-aligned display.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](main.py).

## Native imports for main

In [ ]:
from collections import Counter
from math import fsum, sqrt
from random import Random
import flagquantum as fq
from flagquantum.synth import (
    Output,
    QNum,
    prepare_amplitudes,
    prepare_state,
    qfunc,
    synthesize,
)

## distribution

In [ ]:
@qfunc
def distribution(probabilities: list[float], x: Output[QNum]) -> None:
    prepare_state(probabilities, 0.01, x)

## signed state

In [ ]:
@qfunc
def signed_state(amplitudes: list[float], x: Output[QNum]) -> None:
    prepare_amplitudes(amplitudes, 0, x)

## run example

In [ ]:
def run_example() -> None:
    probabilities = [0.05, 0.11, 0.13, 0.23, 0.27, 0.12, 0.03, 0.06]
    program = synthesize(
        distribution, classical_inputs={"probabilities": probabilities}
    )
    result = fq.run(program)
    named = dict(program.probabilities(result, registers=("x",)))
    samples = Random(17).choices(list(named), weights=list(named.values()), k=1000)
    counts = Counter(samples)
    print("Sampled probabilities:", [counts.get((i,), 0) / 1000 for i in range(8)])

    amplitudes = [-1 + 2 * i / 7 for i in range(8)]
    norm = sqrt(fsum(value * value for value in amplitudes))
    amplitudes = [value / norm for value in amplitudes]
    program = synthesize(
        signed_state, classical_inputs={"amplitudes": amplitudes}, max_workspace=0
    )
    state = fq.run(program).statevector().flatten()
    print("Raw amplitudes:", state.tolist())
    # There is one output register and no hidden storage. Match the source's
    # presentation convention only after retaining the raw signed state.
    global_phase = state[0] / state[0].abs()
    print("Amplitudes up to global phase:", (state / global_phase).real.tolist())

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
run_example()

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states


def householder_reference(amplitudes: list[float]) -> fq.Circuit:
    target = torch.tensor(amplitudes, dtype=torch.complex128)
    target = target / torch.linalg.vector_norm(target)
    difference = -target.clone()
    difference[0] += 1
    reflection = torch.eye(target.numel(), dtype=torch.complex128) - 2 * torch.outer(
        difference, difference.conj()
    ) / torch.vdot(difference, difference)
    width = target.numel().bit_length() - 1
    return fq.Circuit(width, dtype=torch.complex128).any(
        *range(width), matrix=reflection
    )


probabilities = [0.05, 0.11, 0.13, 0.23, 0.27, 0.12, 0.03, 0.06]
compare_states(
    synth.synthesize(distribution, classical_inputs={"probabilities": probabilities}),
    householder_reference([sqrt(value) for value in probabilities]),
    ("x",),
)
amplitudes = [-1 + 2 * index / 7 for index in range(8)]
norm = sqrt(fsum(value * value for value in amplitudes))
amplitudes = [value / norm for value in amplitudes]
compare_states(
    synth.synthesize(
        signed_state, classical_inputs={"amplitudes": amplitudes}, max_workspace=0
    ),
    householder_reference(amplitudes),
    ("x",),
)
print("Independent full-state comparison: passed")
